# 00 - Setup and Validation

This notebook is the shared prerequisite check for **all** demos in the AI
Governance lab series. Run it first, top to bottom, before opening any of the
`demoN-*.ipynb` notebooks.

It will:

1. Confirm you are logged in via `az login` and show the active subscription.
2. Let you choose/confirm the Azure subscription to use.
3. Prompt for (and persist to `.env`) the resource group and APIM instance
   name that the rest of the labs will use.
4. Verify the APIM instance is reachable and print its gateway URL and SKU.

Nothing here creates or modifies any Azure resources -- it only reads
configuration and validates connectivity.


In [1]:
import sys
sys.path.append("..")

from shared import auth, config, display, results


## 1. Confirm Azure CLI login

In [2]:
import json
import subprocess

AZ_ACCOUNT = {}
try:
    # Windows resolves az.cmd only through the shell; Linux and macOS run az directly.
    result = subprocess.run(
        ["az", "account", "show", "-o", "json"],
        capture_output=True, text=True, check=True, timeout=30,
        shell=(sys.platform == "win32"),
    )
    AZ_ACCOUNT = json.loads(result.stdout or "{}")
    display.banner("az login is active.", kind="success")
    print(result.stdout)
except Exception as exc:
    display.banner(
        "Could not confirm az login. Run `az login` in a terminal, then re-run this cell.",
        kind="error",
    )
    raise


{
  "environmentName": "AzureCloud",
  "homeTenantId": "aa93b9d9-037d-4f08-a26d-783cff0e2369",
  "id": "64c3d212-40ed-4c6d-a825-6adfbdf25dad",
  "isDefault": true,
  "managedByTenants": [
    {
      "tenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47"
    },
    {
      "tenantId": "975f013f-7f24-47e8-a7d3-abc4752bf346"
    }
  ],
  "name": "ME-MngEnvMCAP675646-emknafo-1",
  "state": "Enabled",
  "tenantDefaultDomain": "MngEnvMCAP675646.onmicrosoft.com",
  "tenantDisplayName": "Contoso",
  "tenantId": "aa93b9d9-037d-4f08-a26d-783cff0e2369",
  "user": {
    "name": "admin@MngEnvMCAP675646.onmicrosoft.com",
    "type": "user"
  }
}



## 2. Load or collect configuration

In [3]:
cfg = config.load_config(interactive=True)
config.validate_config(cfg)
RESULTS = results.NotebookResults("00-setup-and-validation")
display.header("Current configuration (secrets masked)")
_ = display.show_table([cfg.as_display_dict()])

az_login_ok = bool(AZ_ACCOUNT.get("id"))
RESULTS.record(
    "setup.identity",
    "passed" if az_login_ok and cfg.subscription_id else "failed",
    {
        "az_login": az_login_ok,
        "subscription_configured": bool(cfg.subscription_id),
        "subscription_matches_az_default": AZ_ACCOUNT.get("id") == cfg.subscription_id,
        "user_assigned_identity_configured": bool(cfg.apim_identity_client_id),
    },
)


## Current configuration (secrets masked)

,subscription_id,resource_group,apim_name,aoai_endpoint,aoai_deployment,aoai_key,aoai_api_version,aoai_api_style,app_insights_name,app_insights_resource_id,...,content_safety_threshold_selfharm,content_safety_threshold_sexual,content_safety_threshold_violence,demo4_ptu_east_endpoint,demo4_ptu_central_endpoint,demo4_payg_endpoint,demo4_ptu_east_deployment,demo4_ptu_central_deployment,demo4_payg_deployment,demo_run
0,64c3d212-40ed-4c6d-a825-6adfbdf25dad,rg-aigov-lab,apim-aigov-lab-g01-001-b1b642,https://aif-aigov-lab-g01-001-b1b642.openai.az...,chat,None,2024-10-21,v1,appi-aigov-lab-g01-001,/subscriptions/64c3d212-40ed-4c6d-a825-6adfbdf...,...,4,4,2,,,,,,,112dfa33


'passed'

## 3. Validate the APIM instance is reachable

In [4]:
from shared import apim

service = apim.get_service(cfg.subscription_id, cfg.resource_group, cfg.apim_name)
gateway_url = service["properties"]["gatewayUrl"]
sku = service.get("sku", {}).get("name", "unknown")

display.banner(f"APIM instance '{cfg.apim_name}' is reachable.", kind="success")
display.show_table([
    {
        "name": service.get("name"),
        "location": service.get("location"),
        "sku": sku,
        "gatewayUrl": gateway_url,
        "provisioningState": service.get("properties", {}).get("provisioningState"),
    }
])

supported_llm_token_limit_skus = {
    "Developer", "Basic", "BasicV2", "Standard", "StandardV2", "Premium", "PremiumV2"
}
if sku not in supported_llm_token_limit_skus:
    display.banner(
        "Note: Demo 1 uses llm-token-limit, which is supported on Developer, Basic, "
        "BasicV2, Standard, StandardV2, Premium, and PremiumV2. It is not available "
        "on Consumption. Verify your SKU before running Demo 1.",
        kind="warning",
    )

supported_pool_skus = {"BasicV2", "StandardV2", "PremiumV2", "Standard", "Premium"}
if sku not in supported_pool_skus:
    display.banner(
        "Note: Demo 4 uses backend pools and circuit breakers, which require "
        "BasicV2, StandardV2, PremiumV2, classic Standard, or classic Premium. "
        "They are not available on Consumption, Developer, or classic Basic.",
        kind="warning",
    )

provisioning_state = service.get("properties", {}).get("provisioningState")
gateway_https = str(gateway_url or "").startswith("https://")
RESULTS.record(
    "setup.endpoints",
    "passed" if gateway_https and provisioning_state == "Succeeded" else "failed",
    {
        "apim_reachable": True,
        "gateway_https": gateway_https,
        "provisioning_state": provisioning_state,
        "aoai_endpoint_valid": True,
    },
)
llm_token_limit_supported = sku in supported_llm_token_limit_skus
pool_supported = sku in supported_pool_skus
RESULTS.record(
    "setup.apim_sku",
    "passed" if llm_token_limit_supported and pool_supported else "failed",
    {
        "sku": sku,
        "llm_token_limit_supported": llm_token_limit_supported,
        "pool_supported": pool_supported,
    },
)


ApimError: GET https://management.azure.com/subscriptions/64c3d212-40ed-4c6d-a825-6adfbdf25dad/resourceGroups/rg-aigov-lab/providers/Microsoft.ApiManagement/service/apim-aigov-lab-g01-001-b1b642?api-version=2024-06-01-preview failed with 404: {"error":{"code":"ResourceNotFound","message":"The Resource 'Microsoft.ApiManagement/service/apim-aigov-lab-g01-001-b1b642' under resource group 'rg-aigov-lab' was not found. For more details please go to https://aka.ms/ARMResourceNotFoundFix"}}

## 4. Content Safety resource check (required for Demo 3)


In [ ]:
from shared import config as _config

display.header("Content Safety resource (Demo 3 prerequisite)")

if cfg.content_safety_endpoint:
    try:
        _config.validate_content_safety_config(cfg)
        display.banner(
            f"CONTENT_SAFETY_ENDPOINT is set and well-formed: {cfg.content_safety_endpoint}",
            kind="success",
        )
    except ValueError as exc:
        display.banner(str(exc), kind="error")
    if cfg.content_safety_key:
        display.banner("CONTENT_SAFETY_KEY is set; the managed identity role below is not required.", kind="info")
    else:
        display.banner(
            "No CONTENT_SAFETY_KEY set -- grant the APIM system-assigned managed identity the "
            "'Cognitive Services User' role on the Content Safety resource before running Demo 3 "
            "(Content Safety resource -> Access control (IAM) -> Add role assignment).",
            kind="warning",
        )
else:
    display.banner(
        "CONTENT_SAFETY_ENDPOINT is not set yet. This is only required for Demo 3 -- Demos 1 and 2 "
        "do not need it. Demo 3 will prompt for it (and persist it to .env) when you open "
        "demo3-content-safety.ipynb.",
        kind="warning",
    )


## Next steps

Configuration has been captured and persisted to `.env`. You can now open:

- `demo1-token-limits.ipynb` -- Token limits & quota enforcement
- `demo2-token-metrics.ipynb` -- Token metering & chargeback dimensions
- `demo3-content-safety.ipynb` -- Content safety: inspect both directions
- `demo4-resilient-pool.ipynb` -- Resilient backend pools: priority/weight routing, circuit breakers, spillover
